In [6]:
!pip install mlflow


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('icwsmR_final_dataset_share.csv')
df = df.dropna(subset=['preds_3_label_criteria', 'user_id'])


df['midnight_wt_total'] = df[['hour_0_wt', 'hour_1_wt', 'hour_2_wt', 'hour_3_wt', 'hour_4_wt', 'hour_5_wt']].sum(axis=1)
df['total_wt'] = df['morning_wt'] + df['noon_wt'] + df['afternoon_wt'] + df['evening_wt'] + df['midnight_wt_total']
df['midnight_ratio'] = df['midnight_wt_total'] / (df['total_wt'] + 1)

features_926 = [
    'morning_wt', 'noon_wt', 'afternoon_wt', 'evening_wt',
    'midnight_wt_total', 'total_wt', 'session_all_per_day',
    'category_count_unique', 'age', 'midnight_ratio'
]

target = 'preds_3_label_criteria'

X = df[features_926].fillna(0)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

sample_weights = np.where((X_train['midnight_wt_total'] > 60), 5.0, 1.0)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (80888, 10)
Testing data shape: (20222, 10)


/tmp/ipykernel_3756/139296978.py:5: DtypeWarning: Columns (40) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('icwsmR_final_dataset_share.csv')


In [17]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

mlflow.set_experiment("VADER_Final_Project")

params = {
    "n_estimators": 100,
    "random_state": 42
}

with mlflow.start_run() as run:

    mlflow.log_params(params)

    rf_model = RandomForestClassifier(**params)
    rf_model.fit(X_train, y_train, sample_weight=sample_weights)

    model_info = mlflow.sklearn.log_model(rf_model, "vader_rf_model")

    y_pred = rf_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("macro_f1", macro_f1)

    mlflow.set_tag("Project", "VADER Final Project")
    mlflow.set_tag("Model_Type", "Random Forest")

    print(f"Run ID: {run.info.run_id}")
    print("="*50)
    print(f"Overall Accuracy: {accuracy * 100:.2f}%")
    print("="*50)
    print("\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred))

2026/06/14 15:13:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/14 15:13:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run ID: 8c4284cb7b8543c28470c348b79a691c
Overall Accuracy: 92.67%

Detailed Classification Report:
                   precision    recall  f1-score   support

  Mildly Addicted       0.92      0.88      0.90       714
     Non-Addicted       0.91      0.89      0.90      7051
Severely Addicted       0.94      0.95      0.94     12457

         accuracy                           0.93     20222
        macro avg       0.92      0.91      0.91     20222
     weighted avg       0.93      0.93      0.93     20222



In [19]:
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)

predictions = loaded_model.predict(X_test)
result = pd.DataFrame(X_test, columns=features_926)
result["Actual_Risk"] = y_test
result["MLflow_Prediction"] = predictions

result.head(15)

,morning_wt,noon_wt,afternoon_wt,evening_wt,midnight_wt_total,total_wt,session_all_per_day,category_count_unique,age,midnight_ratio,Actual_Risk,MLflow_Prediction
84407,2596.972003,1010.333333,1893.429293,2895.500000,2705.276302,11101.510931,14.800000,163,52,0.243663,Severely Addicted,Severely Addicted
40523,3118.934921,1089.933333,3148.211996,4738.976496,5129.083333,17225.140079,6.750000,376,57,0.297750,Severely Addicted,Severely Addicted
51291,3794.363636,1484.735043,2191.652778,4216.433333,3878.434343,15565.619133,7.500000,480,26,0.249151,Severely Addicted,Severely Addicted
2787,803.358333,118.900000,282.400000,1172.566558,389.782828,2767.007720,6.307692,201,54,0.140817,Severely Addicted,Severely Addicted
91548,3437.891865,1060.087500,2458.548611,3224.927976,2342.285714,12523.741666,13.535714,412,22,0.187013,Severely Addicted,Severely Addicted
71704,4380.255996,948.200000,2877.420588,3878.312075,2170.850000,14255.038659,11.357143,417,30,0.152276,Severely Addicted,Severely Addicted
15888,4129.500000,1687.000000,1777.000000,4524.500000,2333.333333,14451.333333,10.000000,169,22,0.161450,Severely Addicted,Severely Addicted
33753,89.600000,59.833333,180.700000,141.178571,236.495238,707.807143,8.277778,73,20,0.333652,Severely Addicted,Severely Addicted
67591,138.142857,326.750000,407.250000,1253.775758,0.000000,2125.918615,2.760000,151,49,0.000000,Non-Addicted,Non-Addicted
52416,3600.833333,1266.857143,2277.880647,3187.839044,1989.217414,12322.627582,9.333333,287,26,0.161415,Severely Addicted,Severely Addicted
